In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

DATA = Path("data")
bundle = joblib.load("models/credit_model.joblib")
model, features = bundle["model"], bundle["features"]
cat_cols, cats = bundle["cat_cols"], bundle["categories"]

test_df = pd.read_parquet(DATA / "test_clean.parquet")
for c in cat_cols:
    test_df[c] = pd.Categorical(test_df[c], categories=cats[c])

sample = test_df[features].sample(5000, random_state=42)
contrib = model.predict(sample, pred_contrib=True)   # last column = base value
shap_vals = pd.DataFrame(contrib[:, :-1], columns=features, index=sample.index)

importance = shap_vals.abs().mean().sort_values(ascending=False)
print("Top 15 features by average impact:")
print(importance.head(15).round(4))

Top 15 features by average impact:
EXT_SOURCE_2               0.3190
EXT_SOURCE_3               0.2985
EXT_SOURCE_1               0.1541
AMT_GOODS_PRICE            0.1218
inst_late_rate             0.1173
AMT_CREDIT                 0.1045
DAYS_EMPLOYED              0.1016
ORGANIZATION_TYPE          0.0881
OCCUPATION_TYPE            0.0722
NAME_EDUCATION_TYPE        0.0715
OWN_CAR_AGE                0.0688
prev_refused_rate          0.0677
bureau_total_debt          0.0632
bureau_oldest_loan_days    0.0536
bureau_active_count        0.0449
dtype: float64


In [2]:
p_sample = model.predict_proba(sample)[:, 1]
riskiest = sample.index[p_sample.argmax()]

person = shap_vals.loc[riskiest].sort_values(ascending=False)
print(f"Applicant {test_df.loc[riskiest, 'SK_ID_CURR']}: predicted default risk = {p_sample.max():.1%}")
print(f"Actually defaulted? {'Yes' if test_df.loc[riskiest, 'TARGET'] == 1 else 'No'}\n")

print("Top 5 reasons pushing risk UP:")
for f in person.head(5).index:
    print(f"  {f:25s} value = {sample.loc[riskiest, f]}   impact = +{person[f]:.3f}")

Applicant 280436: predicted default risk = 75.6%
Actually defaulted? No

Top 5 reasons pushing risk UP:
  EXT_SOURCE_3              value = 0.07446068789567313   impact = +1.193
  EXT_SOURCE_2              value = 0.08876193774484964   impact = +0.774
  bureau_active_count       value = 10.0   impact = +0.702
  prev_refused_rate         value = 0.667   impact = +0.298
  bureau_total_debt         value = 930910.5   impact = +0.172


In [3]:
from sklearn.metrics import roc_auc_score

preds = pd.read_parquet(DATA / "test_preds.parquet").set_index("SK_ID_CURR")
test_df["p"] = preds.loc[test_df["SK_ID_CURR"], "p_lgb"].values

cut = test_df["p"].quantile(0.80)
test_df["flagged"] = (test_df["p"] >= cut).astype(int)
test_df["age_group"] = pd.cut(test_df["age_years"], [20, 30, 40, 50, 60, 70],
                              labels=["20-29", "30-39", "40-49", "50-59", "60-69"])

def fairness(col):
    rows = []
    for g, d in test_df.groupby(col, observed=True):
        rows.append({col: g, "people": len(d),
                     "actual_default_%": d["TARGET"].mean() * 100,
                     "avg_predicted_%": d["p"].mean() * 100,
                     "flagged_%": d["flagged"].mean() * 100,
                     "good_payers_flagged_%": d.loc[d["TARGET"] == 0, "flagged"].mean() * 100,
                     "AUC": roc_auc_score(d["TARGET"], d["p"])})
    return pd.DataFrame(rows).round(2)

print(fairness("CODE_GENDER"), "\n")
print(fairness("age_group"))

  CODE_GENDER  people  actual_default_%  avg_predicted_%  flagged_%  \
0           F   40323              7.06             7.26      17.01   
1           M   21179             10.00             9.32      25.70   

   good_payers_flagged_%   AUC  
0                  14.45  0.77  
1                  21.93  0.76   

  age_group  people  actual_default_%  avg_predicted_%  flagged_%  \
0     20-29    8887             11.39            11.53      35.99   
1     30-39   16345              9.48             9.27      25.33   
2     40-49   15383              8.07             7.55      18.02   
3     50-59   13699              5.90             6.25      12.76   
4     60-69    7189              4.94             4.83       6.15   

   good_payers_flagged_%   AUC  
0                  31.63  0.75  
1                  21.71  0.77  
2                  15.15  0.76  
3                  10.79  0.76  
4                   5.24  0.74  


In [4]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

train_df = pd.read_parquet(DATA / "train_clean.parquet")
for c in cat_cols:
    train_df[c] = pd.Categorical(train_df[c], categories=cats[c])

no_age = [f for f in features if f not in ["DAYS_BIRTH", "age_years"]]

X_tr, X_val, y_tr, y_val = train_test_split(
    train_df[no_age], train_df["TARGET"], test_size=0.1,
    stratify=train_df["TARGET"], random_state=42)

m2 = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=31,
                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                        random_state=42, verbose=-1)
m2.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc",
       callbacks=[lgb.early_stopping(100, verbose=False)])

test_df["p_with_age"] = test_df["p"]
test_df["p"] = m2.predict_proba(test_df[no_age])[:, 1]
cut = test_df["p"].quantile(0.80)
test_df["flagged"] = (test_df["p"] >= cut).astype(int)

print("AUC with age:   ", round(roc_auc_score(test_df["TARGET"], test_df["p_with_age"]), 4))
print("AUC without age:", round(roc_auc_score(test_df["TARGET"], test_df["p"]), 4), "\n")
print(fairness("age_group")[["age_group", "flagged_%", "good_payers_flagged_%", "AUC"]])

C:\Users\RAJA VINAY KUMAR\AppData\Roaming\Python\Python312\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


AUC with age:    0.7708
AUC without age: 0.7695 

  age_group  flagged_%  good_payers_flagged_%   AUC
0     20-29      38.38                  34.10  0.75
1     30-39      24.00                  20.45  0.77
2     40-49      17.61                  14.74  0.76
3     50-59      13.03                  11.00  0.76
4     60-69       6.59                   5.71  0.74


## Notebook 07: explaining the model and checking fairness

**What drives predictions (SHAP, 5,000 test applicants):**
The three external scores matter most. Five of my own SQL features made the top 15, with late-payment rate in 5th place.

**One applicant explained:** predicted 75.6% risk, mainly from very low external scores, 10 active loans elsewhere, and 2 of 3 past applications refused. This person actually repaid. At 75.6%, about 1 in 4 people like this still repay.

**Fairness** (flagging the riskiest 20% as a test):
- The model is equally accurate for every gender and age group (AUC 0.74 to 0.77), and its probabilities match reality in each group.
- But good payers get flagged unevenly. Men: 21.9% vs women: 14.5%, even though gender is not an input.
- Age is worse. Good payers aged 20 to 29 get flagged 31.6% of the time, against 5.2% for ages 60 to 69. That is a 6x gap from a real risk gap of only 2.3x.

**I tried removing age.** AUC dropped from 0.7708 to 0.7695 and the 6x gap stayed the same. Other columns, like employment history, carry age anyway. So I kept the original model and will handle this with a human-review band in the next step.